# Appendix B: Data collection

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from time import sleep
from pydaqmx_helper.dac import DAC


ModuleNotFoundError: No module named 'pydaqmx_helper'

In [6]:
# Defining path to folder raw data is saved in
path = "C:/Users/Student/Desktop/Robert-Kelly-Nuvistor-Triode/"

In [7]:
# Interfacing with the USB-6008 and setting the voltage to 0 in the circuit.
# # [From physicslabs.ucd.ie USB-6008 documentation.]
myDAC = DAC(0)
myDAC.writeVoltage(2.5)

#### Setting up the picoammeter

In [8]:
# Import pyserial and set up a connection to the picoammeter.
# # [From physicslabs.ucd.ie RBD 9103 Picoammeter documentation.]
import serial

ser = serial.Serial(
    "com5",
    baudrate = 57600,                    # Sets the bits per second to 57600
    bytesize = serial.EIGHTBITS,         # Sets the data bits to 8
    parity = serial.PARITY_NONE,         # Sets the parity to None
    stopbits = serial.STOPBITS_ONE,      # Sets the stop bits to 1
    xonxoff = False,                     # Sets the flow control to None
    timeout = 1,                         # Sets the timeout to 1 second
)

In [9]:
# Must talk to the picoammeter in arrays of bytes.
# data() turns a string "command" to be sent to the picoammeter into bytes 
# and also ends the string with the newline character "\n" so it does not have to be done manually
# # [From physicslabs.ucd.ie RBD 910 Picoammeter documentation.]

def data(command):
    return bytes(command+"\n", "utf-8")

def communicate(command): # This communicates the command to the picoammeter and returns its answer.
    ser.write(data(command))
    output = ser.readlines()
    print(output)

In [10]:
# Explicitly set filter.
ser.write(data("&F064"))
ser.readlines()

[b'&F, Filter=064\r\n']

In [11]:
ser.write(data("&S"))
ser.readlines()

[b'&S*,Range=002nA,-0.4413,nA\r\n']

#### Define range of voltages to be sent to the circuit

The USB-6008 ranges from 0-5V. We want it to range from negative voltages to positive voltages so the NIDAQ analogue output adapter changes an input of 0V to -5V, 2.5V to 0V, and 5V to 5V.

In [12]:
# Write function to convert the actual voltage we want in the circuit to the value we tell the USB-6008 to transmit.
def convert_voltage(V):
    return V/2 + 2.5

#### Create function to wrangle current measurement into a list of strings


In [13]:
def communicate(command):       # Only valid for commands which return a single line from the picoammeter
    ser.write(data(command))    # When the command is "&S", the picoammeter gives back a list with a single byte inside.
    output = ser.readline()                  # Let output be that list with the single byte.    

    output = output.decode("utf-8").rstrip() # decode the byte into a string and strip the newline character
    output = output.split(",")               # split the string into a list of 4 strings
    
    return output

#### Create function that takes data and saves it for me.

In [1]:
time.time()

NameError: name 'time' is not defined

In [ ]:
def collect_data(V_A_circuit, delay, I_F):
    # V_A_circuit = The array of applied anode voltages actually experienced by the circuit.
    # Delay =       The time in seconds that the code will be delayed before moving to the next voltage in the range.
    # I_F =         The filament current.

    # V_A is the array of applied voltages given to the USB-6008 so that V_A_circuit is actually produced in the circuit.
    V_A = convert_voltage(V_A_circuit)

    # Specify to send voltages using channel 0 on the USB-6008
    myDAC = DAC(0)

    # empty array for recording the anode currents I_A, and their errors.
    I_A = np.array([])
    error_I_A = np.array([])

    # Save all outputs for analysis
    alloutputs = []
    
    # Loop over all voltages in the specified range.
    for V in V_A:

        # Send each voltage in the range to the circuit via the USB-6008
        myDAC.writeVoltage(V)

        if V == V_A[0]: # Sleep for longer before the first measurement
            sleep(10)


        #After each voltage in, wait before moving onto the next voltage in the range.
        sleep(delay)
        
        # Measure current from the picoammeter.
        output = communicate("&S")
        
        #Count number of repetitions needed to get stable measurement. 
        repetitions = 0
        
        print("index = %.f" % np.where(V_A == V) , output[0])
        
         # If picoammeter gives out a bad measurement, print what index of V_A it is at, and 
         # if it over-ranges, under-ranges, or is unstable
        while output[0] != '&S=' and repetitions < 5: # Repeat measurements until the picoammeter gives a stable measurement
                                                        # that is not over-range or under-range.
                                                      # Break loop if repetitions gets very high.                        
            sleep(delay * 2.5)

            output = communicate("&S")

            repetitions += 1

            print(repetitions) 
        
        
        alloutputs = np.append(alloutputs, output)

        # Extracting the current from the output and accounting for the scale.
        if output[3] == "nA":
            # Then the third element of output is the current in nano-amps
            current = float(output[2]) * 10**-9
            error_current = 0.00005 * 10**-9

        if output[3] == "uA":
            # Then the third element of output is the current in micro-amps
            current = float(output[2]) *10**-6
            error_current = 0.00005 * 10**-6

        if output[3] == "mA":
            # Then the third element of output is the current in milli-amps
            current = float(output[2]) *10**-3
            error_current = 0.00005 * 10**-3

        I_A = np.append(I_A, current)
        error_I_A = np.append(error_I_A, error_current)
        print(output)
    myDAC.writeVoltage(convert_voltage(0))

    # Save raw data to a file so it can be accessed later.
    np.save(path+f"anode currents for I_F = {I_F} mA", I_A)
    np.save(path+f"anode voltages for I_F = {I_F} mA", V_A_circuit)
    np.save(path+f"alloutputs for I_F = {I_F} mA", alloutputs)
    np.save(path+f"anode currents errors for I_F = {I_F} mA", error_I_A)
    
# The errors in error_I_A have been found to be mostly incorrect as the units that the picoammeter presents I_A in are
# not good enough. error_I_A actually depends on the range in which I_A is reported.
# New error_I_A is created for each experimental run later in the code

In [ ]:
collect_data(np.arange(-1, 2.5, .005), 2, 105.0)